# 03 Partition Integrity and Roster Audit

This notebook audits the frozen incident-level partitions before raw RouteViews / RIPE RIS MRT acquisition. It does not re-split data, change labels, download MRT files, create Normal labels, introduce a window cap, extract features, train models, or run ablation.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from bgp_incident_detection.audit import (
    PARTITION_DIR,
    completeness_summary,
    duration_summary,
    overall_status,
    print_report,
    run_audit,
)

partition_dir = ROOT / "data/partitions"
partition_dir

## Run Audit and Write Outputs

In [ ]:
results = run_audit(partition_dir)
print_report(results)

## Frozen Count and Manifest Checks

In [ ]:
from collections import Counter

rows = results["rows"]
partition_counts = Counter(row["partition"] for row in rows)
class_partition_counts = Counter((row["partition"], row["category"]) for row in rows)
print(partition_counts)
print(class_partition_counts)
assert partition_counts["train"] == 993
assert partition_counts["validation"] == 213
assert partition_counts["holdout"] == 213
for category in ["Hijack", "Leak", "Outage"]:
    assert class_partition_counts[("train", category)] == 331
    assert class_partition_counts[("validation", category)] == 71
    assert class_partition_counts[("holdout", category)] == 71
assert len({row["incident_uid"] for row in rows}) == 1419

## Known Cross-Schema Raw ID Collisions

In [ ]:
for row in results["cross_schema_collisions"]:
    print(f"{row['incident_id']} | {row['incident_uid']} | {row['category']} | {row['partition']}")
assert len({row["incident_uid"] for row in results["cross_schema_collisions"]}) == len(results["cross_schema_collisions"])

## Duration Summary by Class and Partition

In [ ]:
duration_summary_rows = results["duration_summary"]
for row in duration_summary_rows:
    print(row)

## ASN / Prefix / Source Completeness

In [ ]:
for row in results["completeness_summary"]:
    print(row)

## Diagnostic Overlap Counts

In [ ]:
print("Possible duplicate-event pairs:", len(results["possible_duplicates"]))
print("Cross-partition ASN overlaps:", len(results["asn_overlap"]))
print("Cross-partition prefix overlaps:", len(results["prefix_overlap"]))
print("Cross-partition temporal overlaps:", len(results["temporal_overlap"]))

## Simple Partition Distribution Plots

In [ ]:
try:
    import pandas as pd
    import matplotlib.pyplot as plt

    df = pd.DataFrame(rows)
    df["duration_minutes"] = pd.to_numeric(df["duration_minutes"], errors="coerce")
    df["year"] = pd.to_datetime(df["start_utc"], utc=True).dt.year

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    pd.crosstab(df["partition"], df["category"]).plot(kind="bar", ax=axes[0], title="Class by partition")
    pd.crosstab(df["partition"], df["year"]).plot(kind="bar", ax=axes[1], title="Year by partition")
    df.boxplot(column="duration_minutes", by="partition", ax=axes[2])
    axes[2].set_title("Duration by partition")
    fig.suptitle("")
    plt.tight_layout()
except Exception as exc:
    print(f"Plotting skipped: {exc}")

## PASS / REVIEW / FAIL Summary

In [ ]:
status = overall_status(results["integrity_audit"])
print("Overall status:", status)
for row in results["integrity_audit"]:
    print(row)

print("\nRecommendation: REVIEW before MRT acquisition. Do not re-split, remove incidents, or change labels automatically.")